In [26]:
import os
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("../data/raw/train_2.csv")
df

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,...,"Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.08,6.03,147.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.514774e+07
1,0,2023,1,1.32,6.04,162.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.491475e+07
2,0,2025,1,0.82,6.00,145.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,8.712551e+07
3,0,2025,2,0.90,6.00,118.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,8.265980e+07
4,0,2024,2,1.25,6.06,154.0,16.0,Новый,Большой,Ярославль г,...,73,1,0,0,0,3,1,10,1,7.420934e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485077,21745,2023,10,1.09,6.21,860.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,1.800939e+08
485078,21745,2024,11,0.91,6.21,1226.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,2.074941e+08
485079,21745,2023,11,1.16,6.36,755.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,1.792471e+08
485080,21745,2023,12,1.29,6.97,802.0,9.0,Средний по возрасту,Большой,Павловский Посад г,...,204,5,2,2,4,3,3,13,1,2.163958e+08


In [28]:
# Рабочие часы от 5 до 24
df_cleaned = df.copy()
df_cleaned['Рабочие часы в день'] = df_cleaned['Рабочие часы в день'].clip(5, 25)
df_cleaned['Рабочие часы в день'].value_counts()

Рабочие часы в день
14.0    65390
15.0    64012
13.0    57720
16.0    54600
12.0    47242
17.0    43264
11.0    34658
18.0    30134
10.0    21034
19.0    18694
9.0     12402
20.0    10556
8.0      6552
21.0     6084
22.0     3536
7.0      3094
25.0     1976
23.0     1508
6.0      1066
24.0      806
5.0       754
Name: count, dtype: int64

In [29]:
df_cleaned['Медицинские уч. и аптеки (300 м)'] = df_cleaned['Медицинские уч. и аптеки (300 м)'].clip(0, 10)
df_cleaned['Медицинские уч. и аптеки (300 м)'].value_counts()

Медицинские уч. и аптеки (300 м)
0     183638
1     105768
2      67782
3      45838
4      30108
5      19240
6      11960
7       7956
8       5044
10      4862
9       2886
Name: count, dtype: int64

In [30]:
df_cleaned['Останокви (300 м)'] = df_cleaned['Остановки (300 м)'].clip(0, 15)
df_cleaned['Останокви (300 м)'].value_counts()

Останокви (300 м)
0     267202
2      60996
3      38558
4      34034
1      32396
5      19370
6      12584
7       6396
8       4550
9       2730
10      2002
15      1482
11      1300
12       598
13       546
14       338
Name: count, dtype: int64

In [31]:
df_cleaned['Продуктовые магазины (500 м)'] = df_cleaned['Продуктовые магазины (500 м)'].clip(0, 15)
df_cleaned['Продуктовые магазины (500 м)'].value_counts()

Продуктовые магазины (500 м)
1     67340
2     62062
0     60424
3     59904
4     52676
5     45786
6     39078
7     30238
8     23556
9     16094
10    10920
11     6708
12     4550
13     2704
15     1560
14     1482
Name: count, dtype: int64

In [32]:
df_cleaned['Школы (300 м)'] = df_cleaned['Школы (300 м)'].clip(0, 5)
df_cleaned['Школы (300 м)'].value_counts()

Школы (300 м)
0    337740
1     98930
2     36192
3      8840
4      2444
5       936
Name: count, dtype: int64

In [33]:
df_population = df_cleaned.copy()

In [34]:
# Replace population values with median population for each (city, region) pair, excluding 0 values
# Group by Город and Регион, calculate median population (excluding 0s)
population_medians = df_population.groupby(['Населенный пункт', 'Регион']).apply(
    lambda x: x[x['Численность населения'] != 0]['Численность населения'].median()
).reset_index()
population_medians.columns = ['Населенный пункт', 'Регион', 'Median_Population']
population_medians.fillna(100, inplace=True)

print(f"Population medians for {len(population_medians)} (city, region) pairs:")
print(population_medians.head(10))

# Create a mapping dictionary from (city, region) to median population
population_mapping = dict(zip(
    zip(population_medians['Населенный пункт'], population_medians['Регион']),
    population_medians['Median_Population']
))


# Replace population values
df_population['Численность населения'] = df_population.apply(
    lambda row: population_mapping.get((row['Населенный пункт'], row['Регион']), row['Численность населения']),
    axis=1
)

print(f"\nPopulation values replaced. New distribution:")
print(df_population['Численность населения'].describe())

Population medians for 3375 (city, region) pairs:
                        Населенный пункт              Регион  \
0  1-го отделения совхоза "Масловский" п     Воронежская обл   
1                            1-я Моква д         Курская обл   
2                      Абадзехская ст-ца         Адыгея Респ   
3                                Абаза г        Хакасия Респ   
4                               Абакан г        Хакасия Респ   
5                             Абатское с       Тюменская обл   
6                         Абдрахманово с      Татарстан Респ   
7                             Абдулино г    Оренбургская обл   
8                               Абинск г  Краснодарский край   
9                           Аввакумово д        Тверская обл   

   Median_Population  
0             3545.0  
1              496.0  
2             3147.0  
3            17797.0  
4           164655.0  
5             8241.0  
6             1782.0  
7            18381.0  
8            40964.5  
9              

In [35]:
df_population['Численность населения'].corr(df_population['РТО'])

np.float64(0.3706391018488207)

### Inflation adjustment to march 2025

In [36]:
# Load inflation coefficients for all months from 01(2023) to 03(2025)
# The inflation data for Russian Federation (ИПЦ) is at row 4 (0-indexed), column 2

start_year, start_month = 2023, 1
end_year, end_month = 2025, 3

sheet_names = []

# Generate months and read inflation data
for year in range(start_year, end_year + 1):
    months_range = 12 if year < end_year else end_month
    for month in range(1 if year > start_year else start_month, months_range + 1):
        sheet_name = f"{month:02d}({year})"
        sheet_names.append(sheet_name)

print(f"Total sheets to process: {len(sheet_names)}")
print(f"Sheets range: {sheet_names[0]} to {sheet_names[-1]}")

file_path = "../data/raw/ipc_RF_fo_sub_04-2026.xlsx"
inflation_monthly = {}
errors = []

for sheet_name in sheet_names:
    df_temp = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    # Увеличение цен продовольственных товаров к предыдущему месяцу
    inflation_monthly[sheet_name] = df_temp.iloc[4, 3]
    print(f"{sheet_name}: ИПЦ = {inflation_monthly[sheet_name]}")


print(f"\nSuccessfully loaded {len(inflation_monthly)} months")

Total sheets to process: 27
Sheets range: 01(2023) to 03(2025)
01(2023): ИПЦ = 101.32
02(2023): ИПЦ = 100.79
03(2023): ИПЦ = 100.13
04(2023): ИПЦ = 100.29
05(2023): ИПЦ = 99.69
06(2023): ИПЦ = 99.99
07(2023): ИПЦ = 100.49
08(2023): ИПЦ = 99.94
09(2023): ИПЦ = 100.86
10(2023): ИПЦ = 101.35
11(2023): ИПЦ = 101.55
12(2023): ИПЦ = 101.49
01(2024): ИПЦ = 101.26
02(2024): ИПЦ = 100.77
03(2024): ИПЦ = 100.17
04(2024): ИПЦ = 100.49
05(2024): ИПЦ = 100.41
06(2024): ИПЦ = 100.63
07(2024): ИПЦ = 100.36
08(2024): ИПЦ = 99.99
09(2024): ИПЦ = 100.34
10(2024): ИПЦ = 101.23
11(2024): ИПЦ = 102.33
12(2024): ИПЦ = 102.6
01(2025): ИПЦ = 101.33
02(2025): ИПЦ = 101.27
03(2025): ИПЦ = 100.83

Successfully loaded 27 months


In [37]:
# Calculate coefficients to adjust РТО to March 2025 prices
# The coefficient represents: price_march2025 = price_original * coefficient

# Sort sheet names chronologically (not alphabetically)
def parse_month_year(sheet_name):
    """Parse 'MM(YYYY)' format to (year, month) tuple"""
    month, year = sheet_name.split('(')
    year = int(year.rstrip(')'))
    month = int(month)
    return (year, month)

# Sort chronologically
sheet_names_chrono = sorted(inflation_monthly.keys(), key=parse_month_year)

inflation_coefficients = {}

for i, sheet_name in enumerate(sheet_names_chrono):
    # Start with 1.0 (no adjustment for March 2025 itself)
    coefficient = 1.0
    
    # Multiply by all inflation rates from the month AFTER current to March 2025
    # ИПЦ value represents the change from previous month to current month
    # So to go from current month to March 2025, multiply by all subsequent monthly factors
    for j in range(i + 1, len(sheet_names_chrono)):
        inflation_ipc = inflation_monthly[sheet_names_chrono[j]]
        # Convert ИПЦ (e.g., 100.84) to multiplier (1.0084)
        monthly_multiplier = inflation_ipc / 100.0
        coefficient *= monthly_multiplier

    inflation_prev_month = inflation_monthly[sheet_names_chrono[i]]
    inflation_coefficients[sheet_name] = (coefficient, inflation_prev_month - 100)

print("Inflation adjustment coefficients (multiply РТО by these to get March 2025 prices):\n")
print(f"{'Month':<12} {'Coefficient':<12} {'Increase':<12}")
print("-" * 24)

for sheet_name in sheet_names_chrono:
    coeff = inflation_coefficients[sheet_name]
    print(f"{sheet_name:<12}: {coeff[0]:.5f} {coeff[1]:.5f}")

print(f"\nTotal coefficients calculated: {len(inflation_coefficients)}")
print(f"\nKey values:")
print(f"  01(2023) to 03(2025): {inflation_coefficients['01(2023)'][0]:.8f}")
print(f"  12(2024) to 03(2025): {inflation_coefficients['12(2024)'][0]:.8f}")
print(f"  03(2025) (no change): {inflation_coefficients['03(2025)'][0]:.8f}")

Inflation adjustment coefficients (multiply РТО by these to get March 2025 prices):

Month        Coefficient  Increase    
------------------------
01(2023)    : 1.22673 1.32000
02(2023)    : 1.21712 0.79000
03(2023)    : 1.21554 0.13000
04(2023)    : 1.21202 0.29000
05(2023)    : 1.21579 -0.31000
06(2023)    : 1.21591 -0.01000
07(2023)    : 1.20998 0.49000
08(2023)    : 1.21071 -0.06000
09(2023)    : 1.20039 0.86000
10(2023)    : 1.18440 1.35000
11(2023)    : 1.16632 1.55000
12(2023)    : 1.14919 1.49000
01(2024)    : 1.13490 1.26000
02(2024)    : 1.12622 0.77000
03(2024)    : 1.12431 0.17000
04(2024)    : 1.11883 0.49000
05(2024)    : 1.11426 0.41000
06(2024)    : 1.10729 0.63000
07(2024)    : 1.10331 0.36000
08(2024)    : 1.10342 -0.01000
09(2024)    : 1.09968 0.34000
10(2024)    : 1.08632 1.23000
11(2024)    : 1.06159 2.33000
12(2024)    : 1.03469 2.60000
01(2025)    : 1.02111 1.33000
02(2025)    : 1.00830 1.27000
03(2025)    : 1.00000 0.83000

Total coefficients calculated: 27

K

In [40]:
# Create adjusted РТО column using inflation coefficients
# First, create a month-year identifier from Год and Месяц columns
df_adjusted = df_population.copy()
df_adjusted['month_year'] = df_adjusted['Месяц'].astype(str).str.zfill(2) + '(' + df_adjusted['Год'].astype(str) + ')'

# Apply coefficients to РТО
df_adjusted['РТО_adjusted'] = df_adjusted.apply(
    lambda row: row['РТО'] * inflation_coefficients.get(row['month_year'], (1.0, 0))[0], 
    axis=1
)

print("РТО adjustment completed!")
print(f"\nSample comparison (original vs adjusted РТО):")
print(df_adjusted[['new_id', 'Год', 'Месяц', 'РТО', 'РТО_adjusted']].sort_values(['new_id', 'Год', 'Месяц']).reset_index(drop=True).head(26))

РТО adjustment completed!

Sample comparison (original vs adjusted РТО):
    new_id   Год  Месяц          РТО  РТО_adjusted
0        0  2023      1  74914754.22  9.190022e+07
1        0  2023      2  69240001.40  8.427307e+07
2        0  2023      3  79905726.38  9.712822e+07
3        0  2023      4  78567643.63  9.522558e+07
4        0  2023      5  80856174.96  9.830407e+07
5        0  2023      6  74635062.29  9.074958e+07
6        0  2023      7  73579921.54  8.903038e+07
7        0  2023      8  71929528.80  8.708568e+07
8        0  2023      9  69963064.37  8.398262e+07
9        0  2023     10  76474955.25  9.057661e+07
10       0  2023     11  75835109.27  8.844784e+07
11       0  2023     12  84825173.40  9.748065e+07
12       0  2024      1  75147744.85  8.528481e+07
13       0  2024      2  74209339.11  8.357628e+07
14       0  2024      3  78992997.13  8.881276e+07
15       0  2024      4  78183610.30  8.747414e+07
16       0  2024      5  83985250.56  9.358150e+07
17       

In [41]:
df_adjusted['РТО'] = df_adjusted['РТО_adjusted']
df_adjusted = df_adjusted.drop(columns=['РТО_adjusted'])

### Average spending per region

In [42]:
df_temp = pd.read_excel("../data/raw/DRP_2023_1/spendings.xls", "1.4", header=5)
df_temp

,Unnamed: 0,I квартал 2022 г.,I квартал 2023 г.,I квартал 2022 г..1,I квартал 2023 г..1,I квартал 2022 г..2,I квартал 2023 г..2,I квартал 2022 г..3,I квартал 2023 г..3,I квартал 2022 г..4,...,I квартал 2023 г..9,I квартал 2022 г..10,I квартал 2023 г..10,I квартал 2022 г..11,I квартал 2023 г..11,I квартал 2022 г..12,I квартал 2023 г..12,I квартал 2022 г..13,I квартал 2023 г..13,Unnamed: 29
0,Российская Федерация,22193.429,23220.386,30.977593,32.569971,2.917652,2.842192,6.179185,6.763350,11.532783,...,1.395252,1.622021,1.569548,2.658242,0.849116,1.189280,3.022448,6.829945,2.794023,NaN
1,Центральный\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,федеральный округ,30752.672,27531.660,25.109525,30.684935,2.842117,2.660152,5.251872,7.389485,9.706796,...,0.793701,1.412544,2.132716,2.985315,0.966854,1.249800,3.294825,7.812811,3.108392,NaN
3,Белгородская область,19329.701,20442.569,36.147284,37.322657,3.482201,4.594158,6.114916,5.902585,12.576889,...,0.813577,1.143924,1.334549,3.183717,0.213408,0.197153,2.128255,5.698060,2.100793,NaN
4,Брянская область,15475.415,17170.287,38.620896,38.767581,2.448878,2.978756,6.831487,7.313809,10.711668,...,1.605337,0.988245,0.740535,1.637772,0.001881,2.327091,2.618984,4.822759,2.557721,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,Амурская область,20470.266,18552.181,24.787240,30.622313,2.281426,3.257347,5.420999,6.271586,11.812621,...,0.561584,2.084643,0.959828,1.051173,0.029307,0.769018,0.495634,7.136536,0.172465,NaN
100,Магаданская область,32211.077,38647.251,33.869429,31.067483,3.399293,2.168809,6.122065,4.929256,14.452131,...,1.413992,2.001718,2.272263,2.711055,0.703201,0.326273,1.604049,9.680005,1.467579,NaN
101,Сахалинская область,25077.161,31339.879,34.271994,30.014682,2.391622,1.698877,6.421305,7.535849,12.948643,...,2.745291,3.626495,0.898172,2.339834,1.280155,0.114598,1.166954,7.286176,1.058428,NaN
102,Еврейская автономная область,20186.813,22217.780,30.232335,28.685980,6.434463,5.022613,6.441086,7.181919,13.550935,...,1.906986,1.392706,1.416888,1.134924,0.019781,0.467444,2.070099,7.802207,1.795809,NaN


In [43]:
# Function to normalize region names from DRP to match df_adjusted
# Функция нормализации названий регионов из Excel в формат df_adjusted
def normalize_region_name(region_name):
    if not isinstance(region_name, str):
        return None
    region_name = region_name.strip()
    # Пропускаем строки-заголовки федеральных округов и итогов
    if region_name in ['Российская Федерация', 'федеральный округ', 'Центральный\n', 'Северо-Западный\n ', 'Южный\n', 'Северо-Кавказский\n ', 'Приволжский\n ', 'Уральский\n', 'Сибирский\n', 'Дальневосточный\n']:
        return None
    # Убираем лишние пробелы и символы новой строки
    region_name = region_name.replace('\n', ' ').strip()
    # Обработка городов
    if region_name.startswith('г. '):
        return region_name[3:] + ' г'
    
    # Обработка особых случаев
    if 'Ханты-Мансийский' in region_name:
        return 'Ханты-Мансийский Автономный округ - Югра АО'
    if 'Ямало-Ненецкий' in region_name:
        return 'Ямало-Ненецкий АО'
    # Словарь исключений для сложных случаев
    exceptions = {
        'Республика Северная Осетия-Алания': 'Северная Осетия - Алания Респ',
        'Чувашская Республика': 'Чувашская Республика - Чувашия',
        'Kемеровская область': 'Кемеровская область - Кузбасс обл',
        'Архангельская область без автономного округа': 'Архангельская обл',
        'Тюменская область без автономных округов': 'Тюменская обл',
    }
    if region_name in exceptions:
        return exceptions[region_name]
    # Замена суффиксов
    if region_name.endswith(' область'):
        return region_name[:-7] + 'обл'
    if region_name.endswith(' Республика'):
        return region_name[:-10] + 'Респ'
    if region_name.startswith('Республика '):
        return region_name[11:] + ' Респ'
    if region_name.endswith('край'):
        return region_name  # оставляем как есть
    if 'автономный округ' in region_name:
        # Общий случай для АО
        region_name = region_name.replace('автономный округ', 'АО').strip()
        return region_name
    # Возвращаем как есть, если ничего не подошло
    return None

# Test the function
print("Testing normalize_region_name:")
test_regions = ['Белгородская область', 'Брянская область', 'Московская область']
for region in test_regions:
    print(f"  {region} -> {normalize_region_name(region)}")

Testing normalize_region_name:
  Белгородская область -> Белгородская обл
  Брянская область -> Брянская обл
  Московская область -> Московская обл


In [44]:
# Dictionary mapping DRP files to (year, quarter)
drp_quarters = {
    '../data/raw/DRP_2023_1/spendings.xls': (2023, 1),
    '../data/raw/DRP_2023_2/spendings.xls': (2023, 2),
    '../data/raw/DRP_2023_3/spendings.xls': (2023, 3),
    '../data/raw/DRP_2023_4/spendings.xls': (2023, 4),
    '../data/raw/DRP_2024_1/spendings.xls': (2024, 1),
    '../data/raw/DRP_2024_2/spendings.xls': (2024, 2),
    '../data/raw/DRP_2024_3/spendings.xls': (2024, 3),
    '../data/raw/DRP_2024_4/spendings.xls': (2024, 4),
    '../data/raw/DRP_2025_1/spendings.xls': (2025, 1),
}

all_spendings = []

for file_path, (year, quarter) in drp_quarters.items():
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
    
    try:
        # Read the spendings table
        df_temp = pd.read_excel(file_path, "1.4", header=5)
        
        # Extract region column (first column)
        region_col = df_temp.iloc[:, 0]
        
        # Find the main spending column name (e.g., "I квартал 2023 г.")
        quarter_names = ['I квартал', 'II квартал', 'III квартал', 'IV квартал']
        quarter_num_to_name = {1: 'I квартал', 2: 'II квартал', 3: 'III квартал', 4: 'IV квартал'}
        main_col_pattern = f"{quarter_num_to_name[quarter]} {year} г."
        col_1_pattern = f"{quarter_num_to_name[quarter]} {year} г..1"
        col_2_pattern = f"{quarter_num_to_name[quarter]} {year} г..2"
        
        # Check if required columns exist
        if main_col_pattern not in df_temp.columns or col_1_pattern not in df_temp.columns or col_2_pattern not in df_temp.columns:
            print(f"Required columns not found in {file_path}")
            print(f"  Looking for: {main_col_pattern}, {col_1_pattern}, {col_2_pattern}")
            continue
        
        # Extract main spending and adjustment factors
        main_spending = df_temp[main_col_pattern].values
        factor_1 = df_temp[col_1_pattern].values
        factor_2 = df_temp[col_2_pattern].values
        
        # Calculate adjusted spending: main_spending * (factor_1 + factor_2) / 100
        adjusted_spending = main_spending * (factor_1 + factor_2) / 100
        
        # Create temporary dataframe
        df_spendings = pd.DataFrame({
            'region_drp': region_col.values,
            'year': year,
            'quarter': quarter,
            'spending': adjusted_spending
        })
        
        # Normalize region names
        df_spendings['region'] = df_spendings['region_drp'].apply(normalize_region_name)
        
        # Remove rows with invalid regions
        df_spendings = df_spendings[df_spendings['region'].notna()].copy()
        
        all_spendings.append(df_spendings)
        print(f"Loaded {len(df_spendings)} regions from {year} Q{quarter}")
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Combine all spendings data
df_spendings_all = pd.concat(all_spendings, ignore_index=True)
print(f"\nTotal spendings records: {len(df_spendings_all)}")
print("\nSample of spendings data:")
print(df_spendings_all.head(10))

Loaded 85 regions from 2023 Q1
Loaded 85 regions from 2023 Q2
Loaded 85 regions from 2023 Q3
Loaded 85 regions from 2023 Q4
Loaded 85 regions from 2024 Q1
Loaded 85 regions from 2024 Q2
Loaded 85 regions from 2024 Q3
Loaded 85 regions from 2024 Q4
Loaded 85 regions from 2025 Q1

Total spendings records: 765

Sample of spendings data:
             region_drp  year  quarter  spending            region
0  Белгородская область  2023        1  8568.874  Белгородская обл
1      Брянская область  2023        1  7167.966      Брянская обл
2  Владимирская область  2023        1  7615.972  Владимирская обл
3   Воронежская область  2023        1  6978.540   Воронежская обл
4    Ивановская область  2023        1  7943.582    Ивановская обл
5     Калужская область  2023        1  8221.374     Калужская обл
6   Костромская область  2023        1  7664.800   Костромская обл
7       Курская область  2023        1  8802.128       Курская обл
8      Липецкая область  2023        1  7855.421      Липецка

In [45]:
# Function to map month to quarter
def month_to_quarter(month):
    return (month - 1) // 3 + 1

# Add quarter column to df_adjusted
df_adjusted['quarter'] = df_adjusted['Месяц'].apply(month_to_quarter)

In [46]:
# Добавляем колонку с расходами на продовольствие по региону и кварталу
df_region = df_adjusted.merge(
    df_spendings_all[['year', 'quarter', 'region', 'spending']],
    left_on=['Год', 'quarter', 'Регион'],
    right_on=['year', 'quarter', 'region'],
    how='left'
)

# Переименовываем добавленную колонку
df_region.rename(columns={'spending': 'Траты по региону'}, inplace=True)

# Удаляем временные колонки, появившиеся при слиянии
df_region.drop(columns=['year', 'region', 'quarter'], errors='ignore', inplace=True)

# При необходимости заполняем пропуски медианой по региону (опционально)
# df_region['Траты по региону'] = df_region.groupby('Регион')['Траты по региону'].transform(lambda x: x.fillna(x.median()))
df_region[['Год', "Месяц", "Регион", "Населенный пункт", "РТО", "Траты по региону"]]

,Год,Месяц,Регион,Населенный пункт,РТО,Траты по региону
0,2024,1,Ярославская обл,Ярославль г,8.528481e+07,7796.424
1,2023,1,Ярославская обл,Ярославль г,9.190022e+07,6520.449
2,2025,1,Ярославская обл,Ярославль г,8.896433e+07,9069.495
3,2025,2,Ярославская обл,Ярославль г,8.334588e+07,9069.495
4,2024,2,Ярославская обл,Ярославль г,8.357628e+07,7796.424
...,...,...,...,...,...,...
485077,2023,10,Московская обл,Павловский Посад г,2.133024e+08,9231.537
485078,2024,11,Московская обл,Павловский Посад г,2.202732e+08,11276.429
485079,2023,11,Московская обл,Павловский Посад г,2.090591e+08,9231.537
485080,2023,12,Московская обл,Павловский Посад г,2.486810e+08,9231.537


In [47]:
# No NaNs
df_region[df_region['Траты по региону'].isna()]['Регион']

Series([], Name: Регион, dtype: str)

In [49]:
df_region['Траты по региону'] = df_region.apply(
    lambda row: row['Траты по региону'] * inflation_coefficients.get(row['month_year'], (1.0, 0))[0], 
    axis=1
)

df_region[['Год', "Месяц", "Регион", "Населенный пункт", "РТО", "Траты по региону"]]

,Год,Месяц,Регион,Населенный пункт,РТО,Траты по региону
0,2024,1,Ярославская обл,Ярославль г,8.528481e+07,8848.123532
1,2023,1,Ярославская обл,Ярославль г,9.190022e+07,7998.834269
2,2025,1,Ярославская обл,Ярославль г,8.896433e+07,9260.910410
3,2025,2,Ярославская обл,Ярославль г,8.334588e+07,9144.771808
4,2024,2,Ярославская обл,Ярославль г,8.357628e+07,8780.513577
...,...,...,...,...,...,...
485077,2023,10,Московская обл,Павловский Посад г,2.133024e+08,10933.792952
485078,2024,11,Московская обл,Павловский Посад г,2.202732e+08,11970.921155
485079,2023,11,Московская обл,Павловский Посад г,2.090591e+08,10766.905910
485080,2023,12,Московская обл,Павловский Посад г,2.486810e+08,10608.834279


In [50]:
df_region['Траты по региону'].corr(df_region['РТО'])

np.float64(0.3663819303919754)

In [51]:
df_region = df_region.drop(columns=['month_year'])
df_region.to_csv("../data/processed/v1.csv", index=False)

In [52]:
# ---- 1. Сохраняем коэффициенты инфляции ----
inflation_df = pd.DataFrame([
    {'year': int(key[3:7]), 'month': int(key[:2]), 'inflation_coefficient': val[0], 'inflation_prev_month': val[1]}
    for key, val in inflation_coefficients.items()
])
inflation_df = inflation_df.sort_values(['year', 'month']).reset_index(drop=True)
inflation_df.to_csv('../data/processed/inflation_coefficients.csv', index=False)
print(f"Saved inflation coefficients: {len(inflation_df)} rows")

# ---- 2. Сохраняем траты по регионам (до поправки на инфляцию) ----
# Сохраняем region_spendings после добавления инфляции
region_spending_inflated = df_region[['Год', 'Месяц', 'Регион', 'Траты по региону']].drop_duplicates()
region_spending_inflated = region_spending_inflated.rename(columns={
    'Год': 'year',
    'Месяц': 'month',
    'Регион': 'region',
    'Траты по региону': 'region_spendings_inflated'
}).sort_values(['year', 'month', 'region']).reset_index(drop=True)

region_spending_inflated.to_csv('../data/processed/region_spendings.csv', index=False)
print(f"Saved inflated region spendings: {len(region_spending_inflated)} rows")

Saved inflation coefficients: 27 rows
Saved inflated region spendings: 1742 rows
